#### Data Ingestion to Vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\kaust\AppData\Local\Temp\ipykernel_29144\1867757213.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
print(Path("../data/pdf_files"))

..\data\pdf_files


In [3]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    numPDFs = len(pdf_files)
    print(f"The number of PDF files are {numPDFs}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"This document has {len(documents)} pages")
        
        except Exception as e:
            print(f"Error is {e}")

    print(f"Total Documents : {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")


The number of PDF files are 9
Processing Amity - Handbook.pdf
This document has 80 pages
Processing BennettUniversityHandbook.pdf
This document has 93 pages
Processing Einstein - handbook.pdf
This document has 30 pages
Processing Employee-GNA.pdf
This document has 22 pages
Processing employee-hand-book-updated.pdf
This document has 40 pages
Processing Galgotias - handbook.pdf
This document has 98 pages
Processing IIMA - Handbook.pdf
This document has 208 pages
Processing NID-Handbook.pdf
This document has 34 pages
Processing Reva - handbook.pdf
This document has 52 pages
Total Documents : 657


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 12.6.7 (Build 21G651) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe Acrobat 8.1 Combine Files', 'creationdate': "D:20231031101007Z00'00'", 'author': 'admin', 'moddate': "D:20231101112446Z00'00'", 'title': 'Untitled-1', 'source': '..\\data\\pdf_files\\Amity - Handbook.pdf', 'total_pages': 80, 'page': 0, 'page_label': '1', 'source_file': 'Amity - Handbook.pdf', 'file_type': 'pdf'}, page_content='WELCOME TO AMITYHR MANUAL'),
 Document(metadata={'producer': 'macOS Version 12.6.7 (Build 21G651) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe Acrobat 8.1 Combine Files', 'creationdate': "D:20231031101007Z00'00'", 'author': 'admin', 'moddate': "D:20231101112446Z00'00'", 'title': 'Untitled-1', 'source': '..\\data\\pdf_files\\Amity - Handbook.pdf', 'total_pages': 80, 'page': 1, 'page_label': '2', 'source_file': 'Amity - Handbook.pdf', 'file_type': 'pdf'}, page_content='HR MANUAL\n1. Short Title, Application and Commencement \n 1.1  Sho

#### Text Splitting

In [9]:
def split_documents(documents, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators = ["\n\n", "\n", " ", ""],
        length_function = len
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"{len(documents)} pages have been split into {len(split_docs)} chunks")
    if split_docs:
        print(f"Example chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}...")
    return split_docs

In [13]:
chunks = split_documents(all_pdf_documents, 1000, 200)


657 pages have been split into 1870 chunks
Example chunk: 
Content: WELCOME TO AMITYHR MANUAL...
Metadata: {'producer': 'macOS Version 12.6.7 (Build 21G651) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe Acrobat 8.1 Combine Files', 'creationdate': "D:20231031101007Z00'00'", 'author': 'admin', 'moddate': "D:20231101112446Z00'00'", 'title': 'Untitled-1', 'source': '..\\data\\pdf_files\\Amity - Handbook.pdf', 'total_pages': 80, 'page': 0, 'page_label': '1', 'source_file': 'Amity - Handbook.pdf', 'file_type': 'pdf'}...


In [14]:
chunks

[Document(metadata={'producer': 'macOS Version 12.6.7 (Build 21G651) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe Acrobat 8.1 Combine Files', 'creationdate': "D:20231031101007Z00'00'", 'author': 'admin', 'moddate': "D:20231101112446Z00'00'", 'title': 'Untitled-1', 'source': '..\\data\\pdf_files\\Amity - Handbook.pdf', 'total_pages': 80, 'page': 0, 'page_label': '1', 'source_file': 'Amity - Handbook.pdf', 'file_type': 'pdf'}, page_content='WELCOME TO AMITYHR MANUAL'),
 Document(metadata={'producer': 'macOS Version 12.6.7 (Build 21G651) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe Acrobat 8.1 Combine Files', 'creationdate': "D:20231031101007Z00'00'", 'author': 'admin', 'moddate': "D:20231101112446Z00'00'", 'title': 'Untitled-1', 'source': '..\\data\\pdf_files\\Amity - Handbook.pdf', 'total_pages': 80, 'page': 1, 'page_label': '2', 'source_file': 'Amity - Handbook.pdf', 'file_type': 'pdf'}, page_content='HR MANUAL\n1. Short Title, Application and Commencement \n 1.1  Sho

#### Embedding and VectorDB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLm-L6-V2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    # def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    #     """
    #     Generate embeddings for a list of texts
        
    #     Args:
    #         texts: List of text strings to embed
            
    #     Returns:
    #         numpy array of embeddings with shape (len(texts), embedding_dim)
    #     """
    #     if not self.model:
    #         raise ValueError("Model not loaded")
        
    #     print(f"Generating embeddings for {len(texts)} texts...")
    #     embeddings = self.model.encode(texts, show_progress_bar=True)
    #     print(f"Generated embeddings with shape: {embeddings.shape}")
    #     return embeddings
    

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6147.56it/s]


Model loaded successfully. Embedding dimension: 384


#### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    